In [ ]:
import pandas as pd 

In [ ]:
data_1 = pd.read_csv("bank_customers.csv")
df_1 = data_1.copy()

data_2 = pd.read_csv("bank_loans.csv")
df_2 = data_2.copy()

data_3 = pd.read_csv("bank_payments.csv")
df_3 = data_3.copy()

In [ ]:
print("Customers :")
print(df_1.columns)
print("\nLoans :")
print(df_2.columns)
print("\nLoan Applications :")
print(df_3.columns)

In [ ]:
print("Customers DataFrame :")
print(df_1.isnull().sum())
print("\n")
print("Loans DataFrame :")
print(df_2.isnull().sum())
print("\n")
print("Payments DataFrame :")
print(df_3.isnull().sum())

In [ ]:
# Handling Missing Values in Customers DataFrame

df_1['gender'] = df_1['gender'].fillna(
    df_1['gender'].mode()[0]
)

df_1['annual_income'] = df_1.groupby(
    'employment_type'
)['annual_income'].transform(lambda x: x.fillna(x.median()))

df_1['credit_score'] = df_1.groupby(
    ['state','employment_type']
)['credit_score'].transform(
    lambda x: x.fillna(x.median())
)
print(df_1.isnull().sum())

In [ ]:
# Handling Missing Values in Loans DataFrame

# Merge customer credit score into loan dataset
loan_df = pd.merge(
    df_2,
    df_1[['customer_id', 'credit_score']],
    on='customer_id',
    how='left'
)

# Create credit score categories
loan_df['credit_band'] = pd.cut(
    loan_df['credit_score'],
    bins=[300, 580, 670, 740, 800, 900],
    labels=['Poor', 'Fair', 'Good', 'Very Good', 'Excellent']
)

# Fill interest_rate using median of similar loan type and credit band
loan_df['interest_rate'] = loan_df.groupby(
    ['loan_type', 'credit_band'],
    observed=True
)['interest_rate'].transform(
    lambda x: x.fillna(x.median())
)

# If any values still remain missing, fill using loan type median
loan_df['interest_rate'] = loan_df.groupby(
    'loan_type'
)['interest_rate'].transform(
    lambda x: x.fillna(x.median())
)

# Final fallback: overall median
loan_df['interest_rate'] = loan_df['interest_rate'].fillna(
    loan_df['interest_rate'].median()
)

# Copy cleaned interest_rate back to original loan dataframe
df_2['interest_rate'] = loan_df['interest_rate']

# Check missing values after handling
print("\nMissing Values After:")
print(df_2.isnull().sum())

In [ ]:
# Handling missing values in Customers DataFrame

# Merge loan amount from loan table
df_3 = df_3.merge(
    df_2[['loan_id', 'loan_amount']],
    on='loan_id',
    how='left'
)

# Create loan amount buckets
df_3['loan_bucket'] = pd.qcut(
    df_3['loan_amount'],
    q=5,
    labels=['Very Low', 'Low', 'Medium', 'High', 'Very High']
)

# Fill payment_amount using median of similar loan buckets
df_3['payment_amount'] = df_3.groupby(
    'loan_bucket',
    observed=True
)['payment_amount'].transform(
    lambda x: x.fillna(x.median())
)

# Fallback: overall median
df_3['payment_amount'] = df_3['payment_amount'].fillna(
    df_3['payment_amount'].median()
)

# Fill payment_method using mode
df_3['payment_method'] = df_3['payment_method'].fillna(
    df_3['payment_method'].mode()[0]
)

# Drop helper columns if not needed
df_3.drop(
    columns=['loan_amount', 'loan_bucket']
)

# Check missing values after
print(df_3.isnull().sum())

In [ ]:
# Taking Cleaned Files as Output 

df_1.to_csv("cleaned_bank_customers.csv", index=False)

df_2.to_csv("cleaned_bank_loans.csv", index=False)

df_3.to_csv("cleaned_bank_payments.csv", index=False)